# 🔧 Tool Use Basics

---

## Use case

Les LLMs sont limités à leur connaissance d'entraînement. Les **tools** (ou function calling) permettent de connecter le modèle à des systèmes externes : APIs, bases de données, moteurs de recherche.

Le modèle ne *execute* pas les tools lui-même, il décide *lesquels appeler* et *avec quels arguments*. C'est le développeur qui exécute et renvoie le résultat.

Ce notebook couvre :
- Définir un tool et le passer au modèle
- Cycle complet : appel, exécution, résultat, réponse finale
- Parallel tool calls, quand le modèle appelle plusieurs tools en un échange
- Error handling, signaler un échec au modèle

## Stack

- **OpenAI SDK** (`npm:openai`), client officiel TypeScript
- **AI SDK Vercel** (`npm:ai`, `npm:@ai-sdk/openai`), approche provider-agnostic en fin de notebook

## What's next

- `02-augmentation/rag` , combiner tools et retrieval
- `02-augmentation/mcp-client` , tools via le protocole MCP
- `03-agentique` , orchestrer plusieurs tools dans une boucle agentique

## Setup

**En local**
1. Copier `.env.example` en `.env` à la racine du repo
2. Renseigner `OPENAI_API_KEY`
3. Lancer le notebook avec le kernel Deno

**Sur Google Colab**
> ⚠️ Le kernel Deno n'est pas disponible nativement sur Colab. Ce notebook est prévu pour un environnement local.

In [ ]:
import { load } from "jsr:@std/dotenv";

const env = await load({ envPath: "../../.env" });
const apiKey = env["OPENAI_API_KEY"] ?? Deno.env.get("OPENAI_API_KEY");

if (!apiKey) throw new Error("OPENAI_API_KEY manquante, voir .env.example");

In [ ]:
import OpenAI from "npm:openai";

const client = new OpenAI({ apiKey });

## Définir un tool

Un tool est composé de trois éléments :
- **name** , identifiant appelé par le modèle
- **description** , ce que fait le tool, lu par le modèle pour décider de l'appeler
- **parameters** , schéma JSON des arguments attendus

La description est critique : c'est elle qui guide le modèle dans le choix du bon tool.

1. Définition du tool

In [ ]:
const tools: OpenAI.Chat.ChatCompletionTool[] = [
  {
    type: "function",
    function: {
      name: "web_search",
      description: "Search the web for current information on a given query. Use this when the user asks about recent events or facts that may have changed.",
      parameters: {
        type: "object",
        properties: {
          query: {
            type: "string",
            description: "The search query",
          },
        },
        required: ["query"],
      },
    },
  },
];

2. Mock de la fonction du tool

In [ ]:
interface SearchResult {
  title: string;
  url: string;
  content: string;
}

interface WebSearchResponse {
  query: string;
  results: SearchResult[];
}

function webSearch(query: string): WebSearchResponse {
  return {
    query,
    results: [
      {
        title: "France at the 2026 Winter Olympics",
        url: "https://en.wikipedia.org/wiki/France_at_the_2026_Winter_Olympics",
        content: "France won 3 gold medals, 2 silver medals and 4 bronze medals at the 2026 Winter Olympics.",
      },
    ],
  };
}

## Cycle complet

Le cycle tool use se déroule en quatre étapes :

1. **Appel initial** , on envoie le prompt avec les tools disponibles
2. **Tool call** , le modèle répond avec le tool à appeler et ses arguments
3. **Exécution** , on exécute le tool et on renvoie le résultat au modèle
4. **Réponse finale** , le modèle intègre le résultat et répond à l'utilisateur

1. Appel initial

In [ ]:
const messages: OpenAI.Chat.ChatCompletionMessageParam[] = [
  { role: "user", content: "How many medals did France win at the 2026 Winter Olympics?" },
];

const response = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages,
  tools,
});

console.log("Finish reason:", response.choices[0].finish_reason);
console.log("Response:", response.choices[0].message);

2. Inspecter le tool call

In [ ]:
const toolCall = response.choices[0].message.tool_calls![0];

console.log("Tool appelé:", toolCall.function.name);
console.log("Arguments:", toolCall.function.arguments);

const args = JSON.parse(toolCall.function.arguments) as { query: string };
console.log("Arguments parsés:", args);

3. Exécution du tool et ajout du résultat à la liste de message

In [ ]:
const toolResult = webSearch(args.query);
console.log("Résultat du tool:", toolResult);

messages.push(response.choices[0].message);
messages.push({
  role: "tool",
  tool_call_id: toolCall.id,
  content: JSON.stringify(toolResult),
});

4. Réponse finale du LLM

In [ ]:
const finalResponse = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages,
  tools,
});

console.log("Réponse finale:", finalResponse.choices[0].message.content);

## Parallel tool calls

Le modèle peut décider d'appeler plusieurs tools en un seul échange quand la requête le justifie. On définit deux tools, et on laisse le modèle choisir lesquels appeler.

Le modèle retourne une liste de `tool_calls`, on exécute chacun et on renvoie tous les résultats avant la réponse finale.

1. Initialiser 2 tools

In [ ]:
const toolsParallel: OpenAI.Chat.ChatCompletionTool[] = [
  ...tools,
  {
    type: "function",
    function: {
      name: "get_weather",
      description: "Get the current weather for a given city.",
      parameters: {
        type: "object",
        properties: {
          city: { type: "string", description: "The city name" },
        },
        required: ["city"],
      },
    },
  },
];

interface WeatherResponse {
  city: string;
  temperature: string;
  condition: string;
}

function getWeather(city: string): WeatherResponse {
  return { city, temperature: "12°C", condition: "Partly cloudy" };
}

const toolRegistry: Record<string, (args: Record<string, string>) => unknown> = {
  web_search: ({ query }) => webSearch(query),
  get_weather: ({ city }) => getWeather(city),
};

2. Appel initial parallel

In [ ]:
const messagesParallel: OpenAI.Chat.ChatCompletionMessageParam[] = [
  { role: "user", content: "What's the weather in Milan and how many medals did France win at the 2026 Winter Olympics?" },
];

const responseParallel = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: messagesParallel,
  tools: toolsParallel,
});

console.log("Finish reason:", responseParallel.choices[0].finish_reason);
console.log("Nombre de tool calls:", responseParallel.choices[0].message.tool_calls?.length);

for (const tc of responseParallel.choices[0].message.tool_calls ?? []) {
  console.log(`  Tool: ${tc.function.name}, Arguments: ${tc.function.arguments}`);
}

3. Exécution parallèle des tools


In [ ]:
messagesParallel.push(responseParallel.choices[0].message);

for (const tc of responseParallel.choices[0].message.tool_calls ?? []) {
  const args = JSON.parse(tc.function.arguments) as Record<string, string>;
  const result = toolRegistry[tc.function.name](args);
  console.log(`Résultat ${tc.function.name}:`, result);

  messagesParallel.push({
    role: "tool",
    tool_call_id: tc.id,
    content: JSON.stringify(result),
  });
}

4. Réponse finale du LLM

In [ ]:
const finalResponseParallel = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: messagesParallel,
  tools: toolsParallel,
});

console.log("Réponse finale:", finalResponseParallel.choices[0].message.content);

## Error handling

Si un tool échoue, on ne cache pas l'erreur. On la renvoie au modèle via le message `tool` pour qu'il puisse adapter sa réponse.

Le modèle est capable d'interpréter une erreur et de répondre à l'utilisateur de façon cohérente plutôt que de halluciner un résultat.

1. Echec de l'appel au tool

In [ ]:
function webSearchFailing(_query: string): never {
  throw new Error("Search service unavailable");
}

const messagesError: OpenAI.Chat.ChatCompletionMessageParam[] = [
  { role: "user", content: "How many medals did France win at the 2026 Winter Olympics?" },
];

const responseError = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: messagesError,
  tools,
});

messagesError.push(responseError.choices[0].message);
const tc = responseError.choices[0].message.tool_calls![0];
const args = JSON.parse(tc.function.arguments) as { query: string };

let content: string;
try {
  const result = webSearchFailing(args.query);
  content = JSON.stringify(result);
} catch (e) {
  content = JSON.stringify({ error: (e as Error).message });
  console.log("Erreur capturée:", content);
}

messagesError.push({
  role: "tool",
  tool_call_id: tc.id,
  content,
});

2. Réponse du LLM tenant compte de l'erreur

In [ ]:
const finalResponseError = await client.chat.completions.create({
  model: "gpt-4.1-mini",
  messages: messagesError,
  tools,
});

console.log("Réponse avec erreur:", finalResponseError.choices[0].message.content);

## Vers une approche provider-agnostic avec AI SDK

AI SDK Vercel propose `generateText` avec un paramètre `tools` qui gère le cycle complet automatiquement via `maxSteps`. Plus besoin de gérer manuellement les messages intermédiaires.

1. Config du modèle utilisé

In [ ]:
import { createOpenAI } from "npm:@ai-sdk/openai";
// import { createAnthropic } from "npm:@ai-sdk/anthropic";

const provider = createOpenAI({ apiKey });
const MODEL = provider("gpt-4.1-mini");

// Anthropic, décommenter et renseigner ANTHROPIC_API_KEY dans .env
// const anthropicProvider = createAnthropic({ apiKey: env["ANTHROPIC_API_KEY"] });
// const MODEL = anthropicProvider("claude-haiku-4-5");

2. Cycle complet

In [ ]:
import { generateText, tool } from "npm:ai";
import { z } from "jsr:@zod/zod";

const { text, steps } = await generateText({
  model: MODEL,
  prompt: "How many medals did France win at the 2026 Winter Olympics?",
  maxSteps: 3,
  tools: {
    web_search: tool({
      description: "Search the web for current information on a given query.",
      parameters: z.object({
        query: z.string().describe("The search query"),
      }),
      execute: async ({ query }) => webSearch(query),
    }),
  },
});

console.log("Réponse finale:", text);
console.log("Nombre d'étapes:", steps.length);

for (const step of steps) {
  console.log("Step type:", step.stepType);
}